In [1]:
# ================================================
# HACKER NEWS DATA ANALYSIS
# "What Makes a Post Go Viral?"
# Tool: SQL + BigQuery
# Dataset: bigquery-public-data.hacker_news.full
# Author: Ayush
# ================================================

from google.cloud import bigquery

client = bigquery.Client()
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)

Using Kaggle's public dataset BigQuery integration.


In [2]:
# ================================
# Q1: What are the top 10 viral posts ever?
# ================================

query1 = """
SELECT title, score, `by` AS author
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND score IS NOT NULL
AND title IS NOT NULL
AND `by` IS NOT NULL
ORDER BY score DESC
LIMIT 10
"""

q1 = client.query(query1, job_config=safe_config).to_dataframe()
q1

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,score,author
0,Stephen Hawking has died,6015,Cogito
1,A Message to Our Customers,5771,epaga
2,OpenAI's board has fired Sam Altman,5710,davidbarker
3,Backdoor in upstream xz/liblzma leading to SSH...,4549,rkta
4,CrowdStrike Update: Windows Bluescreen and Boo...,4489,BLKNSLVR
5,Steve Jobs has passed away.,4338,patricktomas
6,Bram Moolenaar has died,4310,wufocaculura
7,Mechanical Watch,4298,todsacerdoti
8,YouTube-dl has received a DMCA takedown from RIAA,4240,phantop
9,Don't post generated/AI-edited comments. HN is...,4229,usefulposter


In [3]:
# ================================
# Q2: Which authors post most consistently?
# ================================

query2 = """
SELECT `by` AS author,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score,
       MAX(score) AS best_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND `by` IS NOT NULL
AND score > 0
GROUP BY `by`
HAVING COUNT(*) >= 10
ORDER BY avg_score DESC
LIMIT 10
"""

q2 = client.query(query2, job_config=safe_config).to_dataframe()
q2

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,author,total_posts,avg_score,best_score
0,km,14,425.50,2358
1,_vvdf,19,367.32,1697
2,phiresky,11,356.09,1812
3,atgctg,12,350.75,2360
4,ssiddharth,16,347.75,2210
5,zachrip,11,330.55,3270
6,CathalMullan,11,327.27,1880
7,rushingcreek,18,321.00,1401
8,tinyprojects,18,313.50,1683
9,natdempk,11,311.45,3393


In [4]:
# ================================
# Q3: What % of content is stories vs comments?
# ================================

query3 = """
SELECT type,
       COUNT(*) AS total,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM `bigquery-public-data.hacker_news.full`
WHERE type IS NOT NULL
GROUP BY type
ORDER BY total DESC
"""

q3 = client.query(query3, job_config=safe_config).to_dataframe()
q3

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,type,total,percentage
0,comment,42520344,87.22
1,story,6192401,12.70
2,job,18163,0.04
3,pollopt,15861,0.03
4,poll,2256,0.00


In [5]:
# ================================
# Q4: Which posts scored above average?
# ================================

query4 = """
SELECT title, score, `by` AS author
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND score > (
    SELECT AVG(score)
    FROM `bigquery-public-data.hacker_news.full`
    WHERE type = "story"
    AND score IS NOT NULL
)
AND title IS NOT NULL
AND `by` IS NOT NULL
ORDER BY score DESC
LIMIT 10
"""

q4 = client.query(query4, job_config=safe_config).to_dataframe()
q4

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,score,author
0,Stephen Hawking has died,6015,Cogito
1,A Message to Our Customers,5771,epaga
2,OpenAI's board has fired Sam Altman,5710,davidbarker
3,Backdoor in upstream xz/liblzma leading to SSH...,4549,rkta
4,CrowdStrike Update: Windows Bluescreen and Boo...,4489,BLKNSLVR
5,Steve Jobs has passed away.,4338,patricktomas
6,Bram Moolenaar has died,4310,wufocaculura
7,Mechanical Watch,4298,todsacerdoti
8,YouTube-dl has received a DMCA takedown from RIAA,4240,phantop
9,Don't post generated/AI-edited comments. HN is...,4229,usefulposter


In [6]:
# ================================
# Q5: How does average score vary by post type?
# ================================

query5 = """
SELECT type,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score,
       MAX(score) AS max_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type IS NOT NULL
AND score IS NOT NULL
GROUP BY type
ORDER BY avg_score DESC
"""

q5 = client.query(query5, job_config=safe_config).to_dataframe()
q5

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,type,total_posts,avg_score,max_score
0,pollopt,15440,39.27,3913
1,poll,2022,26.99,2423
2,story,5928592,13.70,6015
3,job,17440,1.63,220


In [ ]:
# ================================
# Q6: Who are top authors by total score?
# ================================

query6 = """
SELECT `by` AS author,
       COUNT(*) AS total_posts,
       SUM(score) AS total_score,
       ROUND(AVG(score), 2) AS avg_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND `by` IS NOT NULL
AND score > 0
GROUP BY `by`
ORDER BY total_score DESC
LIMIT 10
"""

q6 = client.query(query6, job_config=safe_config).to_dataframe()
q6

In [ ]:
# ================================
# Q7: Which year had the most posts?
# ================================

query7 = """
SELECT EXTRACT(YEAR FROM TIMESTAMP_SECONDS(time)) AS year,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND time IS NOT NULL
AND score IS NOT NULL
GROUP BY year
ORDER BY year ASC
"""

q7 = client.query(query7, job_config=safe_config).to_dataframe()
q7

In [ ]:
# ================================
# Q8: Categorize posts by popularity tier
# ================================

query8 = """
SELECT 
CASE 
    WHEN score >= 1000 THEN "Legendary"
    WHEN score >= 500  THEN "Viral"
    WHEN score >= 100  THEN "Popular"
    WHEN score >= 10   THEN "Normal"
    ELSE "Low"
END AS category,
COUNT(*) AS total_posts
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND score IS NOT NULL
GROUP BY category
ORDER BY total_posts DESC
"""

q8 = client.query(query8, job_config=safe_config).to_dataframe()
q8

In [ ]:
# ================================
# Q9: Authors with highest consistency (low variance)
# ================================

query9 = """
SELECT `by` AS author,
       COUNT(*) AS total_posts,
       ROUND(AVG(score), 2) AS avg_score,
       MAX(score) AS best_score,
       MIN(score) AS worst_score,
       MAX(score) - MIN(score) AS score_range
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
AND `by` IS NOT NULL
AND score > 0
GROUP BY `by`
HAVING COUNT(*) >= 15
ORDER BY avg_score DESC
LIMIT 10
"""

q9 = client.query(query9, job_config=safe_config).to_dataframe()
q9

In [ ]:
# ================================
# Q10: What % of posts are dead or deleted?
# ================================

query10 = """
SELECT
CASE
    WHEN dead = TRUE THEN "Dead"
    WHEN deleted = TRUE THEN "Deleted"
    ELSE "Active"
END AS status,
COUNT(*) AS total,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM `bigquery-public-data.hacker_news.full`
WHERE type = "story"
GROUP BY status
ORDER BY total DESC
"""

q10 = client.query(query10, job_config=safe_config).to_dataframe()
q10

In [7]:
# ================================
# ADVANCED Q1: Elite Post Analysis
# Which posts are true outliers vs just "above average"?
# ================================

query_adv1 = """
WITH story_stats AS (
    -- Calculate baseline average score
    SELECT AVG(score) AS avg_score
    FROM `bigquery-public-data.hacker_news.full`
    WHERE type = "story"
    AND score IS NOT NULL
),
ranked_stories AS (
    SELECT 
        title,
        score,
        `by` AS author,
        -- Rank every post by score globally
        RANK() OVER(ORDER BY score DESC) AS global_rank,
        -- How many times above average is this post?
        ROUND(score / (SELECT avg_score FROM story_stats), 1) AS times_above_avg
    FROM `bigquery-public-data.hacker_news.full`
    WHERE type = "story"
    AND score IS NOT NULL
    AND title IS NOT NULL
    AND `by` IS NOT NULL
)
SELECT 
    global_rank,
    title,
    author,
    score,
    times_above_avg,
    -- Classify post tier based on how far above average
    CASE
        WHEN times_above_avg >= 300 THEN "All Time Legend"
        WHEN times_above_avg >= 200 THEN "Elite"
        WHEN times_above_avg >= 100 THEN "Viral"
        ELSE "Popular"
    END AS tier
FROM ranked_stories
WHERE global_rank <= 10
ORDER BY global_rank ASC
"""

q_adv1 = client.query(query_adv1, job_config=safe_config).to_dataframe()
q_adv1

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,global_rank,title,author,score,times_above_avg,tier
0,1,Stephen Hawking has died,Cogito,6015,439.1,All Time Legend
1,2,A Message to Our Customers,epaga,5771,421.3,All Time Legend
2,3,OpenAI's board has fired Sam Altman,davidbarker,5710,416.9,All Time Legend
3,4,Backdoor in upstream xz/liblzma leading to SSH...,rkta,4549,332.1,All Time Legend
4,5,CrowdStrike Update: Windows Bluescreen and Boo...,BLKNSLVR,4489,327.7,All Time Legend
5,6,Steve Jobs has passed away.,patricktomas,4338,316.7,All Time Legend
6,7,Bram Moolenaar has died,wufocaculura,4310,314.6,All Time Legend
7,8,Mechanical Watch,todsacerdoti,4298,313.8,All Time Legend
8,9,YouTube-dl has received a DMCA takedown from RIAA,phantop,4240,309.5,All Time Legend
9,10,Don't post generated/AI-edited comments. HN is...,usefulposter,4229,308.7,All Time Legend


In [8]:
# ================================
# ADVANCED Q2: Virality Drop-off Analysis
# At which rank does score drop most sharply?
# ================================

query_adv2 = """
WITH top_stories AS (
    SELECT 
        title,
        score,
        `by` AS author,
        RANK() OVER(ORDER BY score DESC) AS rank_num
    FROM `bigquery-public-data.hacker_news.full`
    WHERE type = "story"
    AND score IS NOT NULL
    AND title IS NOT NULL
    AND `by` IS NOT NULL
)
SELECT 
    rank_num,
    title,
    score,
    -- Score of post ranked just above
    LAG(score) OVER(ORDER BY score DESC) AS prev_rank_score,
    -- How much did score drop from previous rank?
    score - LAG(score) OVER(ORDER BY score DESC) AS score_drop,
    -- What % did it drop?
    ROUND(
        (score - LAG(score) OVER(ORDER BY score DESC)) * 100.0 / 
        LAG(score) OVER(ORDER BY score DESC), 1
    ) AS drop_percentage
FROM top_stories
WHERE rank_num <= 10
ORDER BY rank_num ASC
"""

q_adv2 = client.query(query_adv2, job_config=safe_config).to_dataframe()
q_adv2

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,rank_num,title,score,prev_rank_score,score_drop,drop_percentage
0,1,Stephen Hawking has died,6015,<NA>,<NA>,NaN
1,2,A Message to Our Customers,5771,6015,-244,-4.1
2,3,OpenAI's board has fired Sam Altman,5710,5771,-61,-1.1
3,4,Backdoor in upstream xz/liblzma leading to SSH...,4549,5710,-1161,-20.3
4,5,CrowdStrike Update: Windows Bluescreen and Boo...,4489,4549,-60,-1.3
5,6,Steve Jobs has passed away.,4338,4489,-151,-3.4
6,7,Bram Moolenaar has died,4310,4338,-28,-0.6
7,8,Mechanical Watch,4298,4310,-12,-0.3
8,9,YouTube-dl has received a DMCA takedown from RIAA,4240,4298,-58,-1.3
9,10,Don't post generated/AI-edited comments. HN is...,4229,4240,-11,-0.3


In [9]:
# ================================
# ADVANCED Q3: Author Reliability Analysis
# Which authors CONSISTENTLY go viral — not just lucky once?
# ================================

query_adv3 = """
WITH author_stats AS (
    SELECT 
        `by` AS author,
        COUNT(*) AS total_posts,
        ROUND(AVG(score), 1) AS avg_score,
        MAX(score) AS best_score,
        MIN(score) AS worst_score,
        -- Consistency score = gap between best and worst
        MAX(score) - MIN(score) AS score_range,
        -- Rank authors by average score
        RANK() OVER(ORDER BY AVG(score) DESC) AS reliability_rank
    FROM `bigquery-public-data.hacker_news.full`
    WHERE type = "story"
    AND `by` IS NOT NULL
    AND score > 0
    GROUP BY `by`
    -- Must have posted at least 10 times to qualify
    HAVING COUNT(*) >= 10
)
SELECT 
    reliability_rank,
    author,
    total_posts,
    avg_score,
    best_score,
    worst_score,
    -- Smaller range = more consistent
    CASE
        WHEN score_range < 500  THEN "Highly Consistent"
        WHEN score_range < 1000 THEN "Consistent"
        ELSE "Unpredictable"
    END AS consistency_label
FROM author_stats
WHERE reliability_rank <= 10
ORDER BY reliability_rank ASC
"""

q_adv3 = client.query(query_adv3, job_config=safe_config).to_dataframe()
q_adv3

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,reliability_rank,author,total_posts,avg_score,best_score,worst_score,consistency_label
0,1,km,14,425.5,2358,1,Unpredictable
1,2,_vvdf,19,367.3,1697,12,Unpredictable
2,3,phiresky,11,356.1,1812,1,Unpredictable
3,4,atgctg,12,350.7,2360,1,Unpredictable
4,5,ssiddharth,16,347.7,2210,1,Unpredictable
5,6,zachrip,11,330.5,3270,2,Unpredictable
6,7,CathalMullan,11,327.3,1880,1,Unpredictable
7,8,rushingcreek,18,321.0,1401,1,Unpredictable
8,9,tinyprojects,18,313.5,1683,1,Unpredictable
9,10,natdempk,11,311.5,3393,1,Unpredictable


In [ ]:
print("""
KEY INSIGHTS FROM HACKER NEWS ANALYSIS
=======================================

BASIC ANALYSIS:
1. Stephen Hawking's death post = most viral ever (6015 upvotes)
2. Tech controversies & deaths dominate top posts
3. km is the most consistent author (avg 425 score per post)
4. Comments make up ~80% of all content on HN
5. Majority of posts score below 10 — going viral is RARE

ADVANCED ANALYSIS:
6. Stephen Hawking post scored 439x above average — true All Time Legend
7. Biggest virality cliff = rank 3 to rank 4 (-1161 points / -20% drop)
8. Top 3 posts are in a completely different league from rank 4 onwards
9. Most "consistent" authors still have unpredictable score ranges
10. Reliability rank reveals km as the most dependable viral author
""")